In [25]:
PATH_TO_EXPLANATION_IT = "../data/explanations/Imprisonment-IT/"
PATH_TO_EVALUATION_IT = "../data/evaluation/Imprisonment-IT/"
PATH_TO_TEMPLATE = "../data/evaluation/evaluation_form.txt"

MAP_LABELS_IT = {"0": "Released", "1": "NotReleased"}
MAP_LABELS_EN = {"0": "negative", "1": "positive"}

In [26]:
import glob

explanations_files = glob.glob(PATH_TO_EXPLANATION_IT + "*.json")
explanations_files

['../data/explanations/Imprisonment-IT/656.json',
 '../data/explanations/Imprisonment-IT/671.json',
 '../data/explanations/Imprisonment-IT/674.json',
 '../data/explanations/Imprisonment-IT/691.json',
 '../data/explanations/Imprisonment-IT/702.json']

In [27]:
import os
import json
from pathlib import Path

TEMPLATE_EXPLANATION = open(PATH_TO_TEMPLATE, "r").read().strip()


def process_explanation(target: Path, output_folder: Path):
    with open(target, "r") as fp:
        json_explanation = json.load(fp)
    print(json.dumps(json_explanation, indent=3))

    os.makedirs(output_folder, exist_ok=True)

    print("File:", target)
    doc_id = target.stem
    y_true = json_explanation.get('y_pred')
    y_pred = json_explanation.get('y_test')
    content = json_explanation.get('original_content')

    template = str(TEMPLATE_EXPLANATION)

    template = template.replace("@DOC_ID", str(doc_id))
    template = template.replace("@LABEL", MAP_LABELS_IT[str(y_true)])
    template = template.replace("@PREDICTION", MAP_LABELS_IT[str(y_pred)])
    template = template.replace("@TEXT", str(content).strip())

    explanation_per_hypernode = json_explanation["explanation"]
    for hyper_node_explanation_id in list(explanation_per_hypernode)[:2]:
        template_explanation = str(template)  # Copy for this specific explanation

        print("Hyper node: " + hyper_node_explanation_id)
        explanation_content = explanation_per_hypernode[hyper_node_explanation_id]

        words_l0 = explanation_content["words_l0"]
        cg_methods = explanation_content["words_cg_methods"]

        llm_search = cg_methods["llm_search"]
        semantic_search_l0 = cg_methods["semantic_search_l0"]
        semantic_search_l1 = cg_methods["semantic_search_l1"]

        explanation_object = """
### Hypernode ID
@HYPERNODE_ID

### Words highlighted from Layer 0 assigned to this hypernode in Layer 1
@WORDS_L0

### Concept grounding for hypernode in Layer 1

Method 1:
@LLM_SEARCH

Method 2:
@SEMANTIC_SEARCH_L1

Method 3:
@SEMANTIC_SEARCH_L0
        """.strip()

        explanation_object = explanation_object.replace("@HYPERNODE_ID", hyper_node_explanation_id)
        explanation_object = explanation_object.replace("@WORDS_L0", ", ".join(words_l0))

        # Method 1
        explanation_object = explanation_object.replace("@LLM_SEARCH", ", ".join(llm_search))
        # Method 2
        explanation_object = explanation_object.replace("@SEMANTIC_SEARCH_L1", ", ".join(semantic_search_l1))
        # Method 3
        explanation_object = explanation_object.replace("@SEMANTIC_SEARCH_L0", ", ".join(semantic_search_l0))

        template_explanation = template_explanation.replace("@EXPLANATION_OBJECT", explanation_object)

        output_file = Path(
            output_folder) / f"Explanation_Doc-{int(doc_id):03d}_Hypernode_{int(hyper_node_explanation_id):03d}.txt"
        with open(output_file, "w") as fp:
            fp.write(template_explanation)

    # Doc ID

    # Label
    # Prediction

    # Questions
    # 1. Is the prediction correct?


In [28]:
for explanation_file in explanations_files[:3]:
    process_explanation(Path(explanation_file), PATH_TO_EVALUATION_IT)

{
   "explanation": {
      "70": {
         "words_l0": [
            "lo",
            "uno",
            "proprie",
            "sub",
            "stato"
         ],
         "words_cg_methods": {
            "llm_search": [
               "diritto",
               "giustizia",
               "legge"
            ],
            "semantic_search_l0": [
               "stato cui lo",
               "e lo stato",
               "stato lo e"
            ],
            "semantic_search_l1": [
               "ENTITY/Armagh_GAA",
               "ENTITY/Westmeath_GAA",
               "ENTITY/Down_GAA"
            ]
         }
      },
      "139": {
         "words_l0": [
            "art",
            "delinquere",
            "stata",
            "ai",
            "calabria"
         ],
         "words_cg_methods": {
            "llm_search": [
               "diritto penale",
               "giustizia",
               "legislazione"
            ],
            "semantic_search_l0": [
    